# FLAb Thermostability: ESM2 + MLP (Heavy-Only)

This notebook is a clean wrapper around `train_flab_esm2_mlp_heavy_only.py`.
It uses only heavy-chain embeddings, trains an MLP regressor, and reports non-holdout vs holdout metrics.


## 1) Setup

Expected files:
- `flab_thermo_unified_ml_tm_only.csv`
- `output/esm_cache/esm2_space_cache_separate_hl.npz`
- `output/esm_cache/esm2_space_cache_separate_hl_map.json`
- `train_flab_esm2_mlp_heavy_only.py`


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

DATA_CSV = Path('flab_thermo_unified_ml_tm_only.csv')
CACHE_NPZ = Path('output/esm_cache/esm2_space_cache_separate_hl.npz')
CACHE_MAP = Path('output/esm_cache/esm2_space_cache_separate_hl_map.json')
SCRIPT = Path('train_flab_esm2_mlp_heavy_only.py')
OUT_DIR = Path('output/esm_results_heavy_only')

for p in [DATA_CSV, CACHE_NPZ, CACHE_MAP, SCRIPT]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print('All required files found.')


## 2) Run training

The split logic is in the script:
- hold out one full experiment (`rosace2023automated_tm1_golimumab.csv`) for final generalization check
- split every other experiment by unique heavy sequence into 80/10/10 (train/val/test)


In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT),
    '--data_csv', str(DATA_CSV),
    '--cache_npz', str(CACHE_NPZ),
    '--cache_map', str(CACHE_MAP),
    '--holdout_experiment', 'rosace2023automated_tm1_golimumab.csv',
    '--train_frac', '0.80',
    '--val_frac', '0.10',
    '--test_frac', '0.10',
    '--save_plots',
    '--out_dir', str(OUT_DIR),
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Training script failed.')


## 3) Inspect metrics


In [ ]:
metrics_by_seed = pd.read_csv(OUT_DIR / 'metrics_by_seed.csv')
metrics_summary = pd.read_csv(OUT_DIR / 'metrics_summary.csv', index_col=0)

display(metrics_by_seed)
display(metrics_summary)

print('Mean non-holdout test R2:', round(metrics_by_seed['test_r2'].mean(), 4))
print('Mean holdout R2:', round(metrics_by_seed['hold_r2'].mean(), 4))


## 4) Inspect plots

`loss_curve.png` should show train/val behavior.
`pred_vs_true.png` shows test and holdout predictions for seed 1.


In [ ]:
loss_png = OUT_DIR / 'loss_curve.png'
pred_png = OUT_DIR / 'pred_vs_true.png'

for p in [loss_png, pred_png]:
    if not p.exists():
        raise FileNotFoundError(f'Missing plot: {p}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, p, title in zip(axes, [loss_png, pred_png], ['Loss Curve', 'Predicted vs True']):
    ax.imshow(plt.imread(p))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()


## 5) Notes

- Holdout metrics can be unstable when holdout `n` is tiny (here often 5 rows).
- For more reliable generalization estimates, choose a larger holdout experiment.
- If you want faster iteration, temporarily disable `--save_plots`.
